# Results Chapter Draft

Converted from the original Python workflow script so the dissertation repository uses notebook-based workflow artefacts.


In [ ]:
from pathlib import Path
import csv

from docx import Document
from docx.enum.table import WD_ALIGN_VERTICAL
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor


ROOT = Path("/Users/lu_nanxi/CASA/Dissertation_Data")
MODEL_DIR = ROOT / "Vivacity_full_day_cutoff_20260526" / "modelling_ready" / "models"
EQUITY_DIR = ROOT / "Vivacity_full_day_cutoff_20260526" / "equity_context"
OUT = ROOT / "dissertation_results_chapter_draft_vivacity.docx"


def read_csv(path):
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def set_cell_shading(cell, fill):
    tc_pr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:fill"), fill)
    tc_pr.append(shd)


def set_cell_margins(cell, top=80, start=110, bottom=80, end=110):
    tc = cell._tc
    tc_pr = tc.get_or_add_tcPr()
    tc_mar = tc_pr.first_child_found_in("w:tcMar")
    if tc_mar is None:
        tc_mar = OxmlElement("w:tcMar")
        tc_pr.append(tc_mar)
    for m, v in [("top", top), ("start", start), ("bottom", bottom), ("end", end)]:
        node = tc_mar.find(qn(f"w:{m}"))
        if node is None:
            node = OxmlElement(f"w:{m}")
            tc_mar.append(node)
        node.set(qn("w:w"), str(v))
        node.set(qn("w:type"), "dxa")


def set_table_borders(table):
    tbl_pr = table._tbl.tblPr
    borders = tbl_pr.first_child_found_in("w:tblBorders")
    if borders is None:
        borders = OxmlElement("w:tblBorders")
        tbl_pr.append(borders)
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        elem = borders.find(qn(f"w:{edge}"))
        if elem is None:
            elem = OxmlElement(f"w:{edge}")
            borders.append(elem)
        elem.set(qn("w:val"), "single")
        elem.set(qn("w:sz"), "4")
        elem.set(qn("w:space"), "0")
        elem.set(qn("w:color"), "DADCE0")


def add_run(paragraph, text, bold=False, italic=False, size=11, color="000000"):
    run = paragraph.add_run(text)
    run.bold = bold
    run.italic = italic
    run.font.name = "Arial"
    run._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
    run.font.size = Pt(size)
    run.font.color.rgb = RGBColor.from_string(color)
    return run


def add_para(doc, text):
    p = doc.add_paragraph()
    p.paragraph_format.space_after = Pt(8)
    p.paragraph_format.line_spacing = 1.15
    add_run(p, text)
    return p


def add_heading(doc, text, level=1):
    p = doc.add_paragraph()
    p.paragraph_format.space_before = Pt(18 if level == 1 else 13)
    p.paragraph_format.space_after = Pt(6)
    add_run(p, text, size=19 if level == 1 else 14, color="000000")
    return p


def add_bullets(doc, items):
    for text in items:
        p = doc.add_paragraph(style="List Bullet")
        p.paragraph_format.space_after = Pt(4)
        p.paragraph_format.line_spacing = 1.15
        add_run(p, text)


def pct(x):
    return f"{float(x):+.2f}%"


def pval(x):
    value = float(x)
    if value < 0.001:
        return "<0.001"
    return f"{value:.3f}"


def direction_label(value):
    return value.replace("_", " ").replace("statistically flagged", "statistically flagged")


def build_effect_lookup(rows):
    lookup = {}
    for row in rows:
        lookup[(row["scheme_id"], row["outcome_label"], row["term"])] = row
    return lookup


def add_effect_table(doc, headline_rows):
    lookup = build_effect_lookup(headline_rows)
    table = doc.add_table(rows=1, cols=6)
    table.autofit = False
    widths = [0.65, 1.05, 1.35, 0.65, 1.45, 1.25]
    for i, width in enumerate(widths):
        table.columns[i].width = Inches(width)
    set_table_borders(table)
    headers = ["Scheme", "Outcome", "Immediate level", "p", "Slope change", "Reading"]
    for i, header in enumerate(headers):
        cell = table.rows[0].cells[i]
        set_cell_shading(cell, "F1F3F4")
        set_cell_margins(cell)
        add_run(cell.paragraphs[0], header, bold=True, size=8.5)

    for scheme in ["12d", "12f", "13"]:
        for outcome in ["Active travel", "Pedestrian", "Cyclist"]:
            level = lookup[(scheme, outcome, "treated_postTRUE")]
            slope = lookup[(scheme, outcome, "treated_post_time_weeks")]
            reading = (
                "flagged slope"
                if "statistically_flagged" in slope["direction"]
                else "uncertain"
            )
            values = [
                scheme,
                outcome,
                pct(level["percent_change"]),
                pval(level["p_value"]),
                f"{pct(slope['percent_change'])}/week; p={pval(slope['p_value'])}",
                reading,
            ]
            cells = table.add_row().cells
            for i, value in enumerate(values):
                set_cell_margins(cells[i])
                cells[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
                p = cells[i].paragraphs[0]
                p.paragraph_format.space_after = Pt(0)
                if i in [0, 3, 5]:
                    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
                add_run(p, value, size=8.3)


def add_equity_table(doc, rows):
    keep = [r for r in rows if r["analysis_scheme_id"] in ["12d", "12f", "13"]]
    table = doc.add_table(rows=1, cols=7)
    table.autofit = False
    widths = [0.65, 0.85, 1.55, 1.05, 1.15, 1.05, 1.25]
    for i, width in enumerate(widths):
        table.columns[i].width = Inches(width)
    set_table_borders(table)
    headers = [
        "Scheme",
        "Role",
        "LSOA",
        "IMD band",
        "Mean IMD",
        "No car %",
        "Walk/bike commute %",
    ]
    for i, header in enumerate(headers):
        cell = table.rows[0].cells[i]
        set_cell_shading(cell, "F1F3F4")
        set_cell_margins(cell)
        add_run(cell.paragraphs[0], header, bold=True, size=8.2)
    for row in keep:
        active_commute = float(row["mean_pct_commute_bicycle"]) + float(row["mean_pct_commute_on_foot"])
        values = [
            row["analysis_scheme_id"],
            row["dataset_role"],
            row["lsoas"],
            row["imd_bands"].replace("More deprived ", "Deprived ").replace("Middle deprivation ", "Middle "),
            f"{float(row['mean_imd_decile']):.1f}",
            f"{float(row['mean_pct_households_no_car']):.1f}",
            f"{active_commute:.1f}",
        ]
        cells = table.add_row().cells
        for i, value in enumerate(values):
            set_cell_margins(cells[i])
            cells[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
            p = cells[i].paragraphs[0]
            p.paragraph_format.space_after = Pt(0)
            if i in [0, 1, 4, 5, 6]:
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            add_run(p, value, size=8.0)


def add_callout(doc, title, text):
    table = doc.add_table(rows=1, cols=1)
    set_table_borders(table)
    cell = table.rows[0].cells[0]
    set_cell_shading(cell, "F8F9FA")
    set_cell_margins(cell, top=120, start=140, bottom=120, end=140)
    p = cell.paragraphs[0]
    p.paragraph_format.space_after = Pt(2)
    add_run(p, title + ": ", bold=True, size=10)
    add_run(p, text, size=10)


def build_doc():
    headline_rows = read_csv(MODEL_DIR / "vivacity_exploratory_headline_effects.csv")
    equity_rows = read_csv(EQUITY_DIR / "vivacity_scheme_role_equity_context_summary.csv")

    doc = Document()
    section = doc.sections[0]
    section.top_margin = Inches(1)
    section.bottom_margin = Inches(1)
    section.left_margin = Inches(1)
    section.right_margin = Inches(1)

    styles = doc.styles
    styles["Normal"].font.name = "Arial"
    styles["Normal"]._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
    styles["Normal"].font.size = Pt(11)

    title = doc.add_paragraph()
    title.paragraph_format.space_after = Pt(3)
    add_run(title, "Draft Results Chapter: Vivacity Active Travel Analysis", size=25)

    subtitle = doc.add_paragraph()
    subtitle.paragraph_format.space_after = Pt(12)
    add_run(
        subtitle,
        "Working prose for dissertation Results chapter, based on data cleaned to 26 May 2026",
        size=11,
        color="555555",
    )

    add_heading(doc, "4.1 Introduction", 1)
    add_para(
        doc,
        "This chapter presents the empirical findings from the Vivacity countline analysis. The purpose is to assess whether walking and cycling trajectories around selected active travel schemes changed after intervention, and whether the available evidence allows comparison between more deprived and more affluent neighbourhood contexts.",
    )
    add_para(
        doc,
        "The analysis should be read as an exploratory matched-control time-series study. Exact scheme dates are now available and should be used directly as the intervention dates in the next model run. The results therefore indicate relative trajectory differences around the intervention period, rather than definitive causal effects.",
    )
    add_para(
        doc,
        "The model intervention dates are: 12b = 2023-03-01, 12d = 2023-03-01, 12e = 2023-03-01, 12f = 2023-06-01, and 13 = 2024-03-01.",
    )

    add_heading(doc, "4.2 Data Inclusion and Analytical Scope", 1)
    add_para(
        doc,
        "All Vivacity observations were filtered to complete days up to 26 May 2026. A pre-modelling integrity gate then excluded rows before each countline's first reliable date and removed low-availability or error-flagged days. Controls were eligible for exploratory causal comparison only if they had a reliable pre-intervention baseline and sufficient post-intervention coverage.",
    )
    add_para(
        doc,
        "This gate narrowed the main treated-control modelling to schemes 12d, 12f, and 13. Schemes 12b and 12e were retained for descriptive interpretation in the previous outputs, but their status should now be reassessed using confirmed exact scheme dates and first reliable sensor dates.",
    )
    add_callout(
        doc,
        "Writing point",
        "This is important for the dissertation argument: the data structure itself limits the strength of inference, so the results chapter should foreground transparency about inclusion rather than trying to force all schemes into one causal model.",
    )

    add_heading(doc, "4.3 Deprivation and Affluence Context", 1)
    add_para(
        doc,
        "The final causal modelling set is not evenly distributed between deprived and affluent neighbourhoods. Across the 25 causal countlines, 19 are located in more deprived LSOAs, while only three are in more affluent LSOAs. This means that the analysis can describe the socio-economic context of scheme and control locations, but it cannot robustly estimate a deprivation-by-intervention interaction.",
    )
    add_equity_table(doc, equity_rows)
    add_para(
        doc,
        "Scheme 12d compares treated and control countlines that are both in more deprived Liverpool LSOAs. Scheme 13 is also strongly concentrated in more deprived Halton LSOAs. Scheme 12f is the clearest contrast in the current dataset, with treated countlines in a middle-deprivation Wirral LSOA and controls in a more affluent Wirral LSOA. However, because there are only three treated countlines and three main controls for 12f, this should be interpreted as contextual evidence rather than a robust equity effect estimate.",
    )

    add_heading(doc, "4.4 Exploratory Treated-Control Model Results", 1)
    add_para(
        doc,
        "The exploratory models compare weekly treated and control trajectories after intervention, controlling for time trend, post-intervention period, seasonality, and countline fixed effects. The main terms are the immediate treated-control level change after intervention and the additional treated-control weekly slope change after intervention.",
    )
    add_effect_table(doc, headline_rows)
    add_para(
        doc,
        "Most estimated treated-control differences are statistically uncertain. The exceptions are negative post-intervention slope terms for scheme 12d cyclist counts and scheme 12f active travel counts. These should be treated as signals that the treated countlines did not outperform their matched controls during the observed period, rather than as definitive evidence that the schemes reduced active travel.",
    )

    add_heading(doc, "4.5 Scheme-Level Interpretation", 1)
    add_para(
        doc,
        "For scheme 12d, the active travel and pedestrian outcomes do not show clear treated-control divergence. Cyclist counts show an uncertain positive immediate change, followed by a statistically flagged negative relative slope. In plain terms, cycling at treated countlines may have weakened relative to controls after the intervention month, but the interpretation remains sensitive to the first-day-of-installation-month dating assumption and control-site validity.",
    )
    add_para(
        doc,
        "For scheme 12f, the active travel model indicates a statistically flagged negative post-intervention slope relative to controls. Pedestrian and cyclist outcomes are also negative in direction, although statistically uncertain. This is the most relevant scheme for the dissertation's deprivation-context question because the treated site and control site sit in different IMD bands, but the small countline pool means the result should not be presented as a general deprived-versus-affluent conclusion.",
    )
    add_para(
        doc,
        "For scheme 13, the immediate level estimates are negative but uncertain for active travel, pedestrians, and cyclists. The post-intervention slope terms are also statistically uncertain. The evidence is therefore inconclusive: the model does not show a clear uplift in active travel relative to controls, but neither does it provide a stable estimate of negative impact.",
    )

    add_heading(doc, "4.6 Answering the Research Question", 1)
    add_para(
        doc,
        "The research question asks whether deprived and affluent neighbourhoods exhibit different time-series trajectories in walking and cycling uptake after active travel interventions, compared with matched non-intervention areas. At this stage, the answer is necessarily qualified. The data support an exploratory comparison of trajectories around three schemes, but the available causal modelling sample does not provide enough socio-economic variation to make a strong deprivation-versus-affluence causal claim.",
    )
    add_para(
        doc,
        "The strongest finding is therefore methodological and empirical: deprived neighbourhood contexts are well represented in the usable treated-control dataset, while affluent contexts are comparatively sparse. Within the schemes that can be modelled, there is no clear evidence of a consistent post-intervention increase in walking or cycling relative to controls. The statistically flagged terms point in a negative direction for 12d cycling and 12f active travel, but these should be framed as exploratory signals requiring further site verification and sensitivity testing.",
    )

    add_heading(doc, "4.7 Limitations to Carry Forward", 1)
    add_bullets(
        doc,
        [
            "Confirmed scheme dates are encoded in the current workflow; the current report and diagnostics use them.",
            "Some treated sensors begin producing reliable records after the confirmed intervention date, which limits pre-intervention analysis.",
            "Control countlines require manual verification to rule out contamination from other transport or public realm interventions.",
            "The number of schemes and valid controls is too small for a robust deprivation-stratified causal model.",
            "Sensor counts capture flows at specific countlines, not all active travel behaviour within the wider neighbourhood.",
        ],
    )

    add_heading(doc, "4.8 Provisional Chapter Conclusion", 1)
    add_para(
        doc,
        "Overall, the Vivacity analysis currently provides a transparent exploratory account of active travel trajectories around selected Liverpool City Region schemes. It does not yet demonstrate that interventions generated increased walking or cycling uptake relative to matched controls. Nor does it establish a robust deprived-versus-affluent difference in intervention response. Instead, it shows that the usable evidence base is uneven: several modelled comparisons are concentrated in more deprived areas, while affluent comparison cases are limited. This unevenness is itself substantively important for interpreting the equity dimension of active travel monitoring.",
    )

    doc.save(OUT)


if __name__ == "__main__":
    build_doc()
    print(OUT)
